In [2]:
# Install ultralytics if not already present on the server
# !pip install ultralytics pandas matplotlib

from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import os
from IPython.display import display

In [8]:
# UPDATE THESE PATHS TO MATCH YOUR SERVER DIRECTORIES
WEIGHTS_PATH = './yolo 50 epochs/best.pt'  # Path to your trained model weights
DATA_YAML_PATH = '3/data.yaml'                # Path to your dataset config file

In [9]:
print(f"Loading model from: {WEIGHTS_PATH}")
model = YOLO(WEIGHTS_PATH)

Loading model from: ./yolo 50 epochs/best.pt


In [13]:
import os

# 1. Target the exact file inside the '3' folder
yaml_path = '3/data.yaml'

# 2. Get the full, absolute path of the '3' folder on your server
absolute_path_to_3 = os.path.abspath('3')

# 3. Read the file contents
with open(yaml_path, 'r') as f:
    lines = f.readlines()

# 4. Swap out the broken Windows path line with the correct server path
new_lines = []
for line in lines:
    if line.strip().startswith('path:'):
        new_lines.append(f"path: {absolute_path_to_3}\n")
    # Just in case, convert any accidental Windows backslashes in image paths to forward slashes
    elif line.strip().startswith(('train:', 'val:', 'test:')):
        new_lines.append(line.replace('\\', '/'))
    else:
        new_lines.append(line)

# 5. Save it back to the file
with open(yaml_path, 'w') as f:
    f.writelines(new_lines)

print(f"Successfully fixed {yaml_path}!")
print(f"The path inside the file is now set to: {absolute_path_to_3}")

Successfully fixed 3/data.yaml!
The path inside the file is now set to: /home/jovyan/3


In [14]:
# Run comprehensive evaluation on the validation set.
# The 'plots=True' argument forces the generation of confusion matrices and PR curves.
print("Starting evaluation...")
metrics = model.val(
    data=DATA_YAML_PATH,
    split='val', # Change to 'test' if evaluating on a held-out test set
    conf=0.25,   # Confidence threshold
    iou=0.6,     # NMS IOU threshold
    plots=True,
    device=0     # Uses GPU 0. Change to 'cpu' if no GPU is available
)

Starting evaluation...
Ultralytics 8.4.92 🚀 Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10823MiB)
WARNING ⚠️ val: Slow image access detected (ping: 0.1±0.0 ms, read: 8.0±3.0 MB/s, size: 411.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /home/jovyan/3/labels/test_split.cache... 100 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 100/100 9.8Mit/s 0.0s
val: /home/jovyan/3/images/test_split/frame_00173.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 7. Possible class labels are 0-6
val: /home/jovyan/3/images/test_split/frame_00194.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 7. Possible class labels are 0-6
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.2it/s 3.2s0.2ss
                   all         98        688      0.873  

In [15]:
print("--- Overall Validation Metrics ---")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@75:    {metrics.box.map75:.4f}")

# The validation run automatically saves plots (confusion matrix, PR curves) to a directory.
print(f"\nDetailed plots saved to: {metrics.save_dir}")

--- Overall Validation Metrics ---
mAP@50-95: 0.5684
mAP@50:    0.7452
mAP@75:    0.6300

Detailed plots saved to: /home/jovyan/runs/detect/val-7


In [16]:
# Extract class names and their respective performance to diagnose specific classes
class_names = metrics.names
maps_50_95 = metrics.box.maps # mAP@50-95 for each class

# Create a DataFrame for cleaner viewing
metrics_df = pd.DataFrame({
    'Class ID': list(class_names.keys()),
    'Class Name': list(class_names.values()),
    'mAP@50-95': maps_50_95
})

print("\n--- Class-Specific Performance (Worst to Best) ---")
# Sorting ascending to immediately see the bottleneck classes at the top
display(metrics_df.sort_values(by='mAP@50-95', ascending=True))


--- Class-Specific Performance (Worst to Best) ---


,Class ID,Class Name,mAP@50-95
6,6,road_vehicle,0.297750
5,5,person,0.524749
3,3,signal_pole,0.527550
0,0,class_0,0.568425
7,7,bicycle,0.568425
4,4,signal,0.631672
2,2,catenary_pole,0.671406
1,1,train,0.757423


In [17]:
# Run inference on a batch of validation images to visually inspect False Negatives
# Update this path to your actual validation images directory
VAL_IMAGES_DIR = 'data/images/val' 

if os.path.exists(VAL_IMAGES_DIR):
    print("Running visual inference on validation set...")
    predictions = model.predict(
        source=VAL_IMAGES_DIR,
        conf=0.25,
        save=True,
        show_labels=True,
        show_conf=True
    )
    print(f"Visualizations saved to: {predictions[0].save_dir}")
else:
    print(f"Directory {VAL_IMAGES_DIR} not found. Update the path to run visual inference.")

Directory data/images/val not found. Update the path to run visual inference.


In [19]:
import pandas as pd
from IPython.display import display

# 1. Pull the model's class names mapping
class_names = metrics.names

# 2. Get the indices of classes that actually appeared in this validation run
evaluated_classes = metrics.box.ap_class_index

# 3. Pull the arrays (precisions and recalls align with evaluated_classes)
precisions = metrics.box.p
recalls = metrics.box.r
maps_50_95 = metrics.box.maps  # This covers all configured classes

# 4. Safely construct matching rows only for evaluated items
rows = []
for i, class_id in enumerate(evaluated_classes):
    rows.append({
        'Class ID': class_id,
        'Class Name': class_names.get(class_id, f"Class_{class_id}"),
        'Precision (P)': precisions[i],
        'Recall (R)': recalls[i],
        'mAP@50-95': maps_50_95[class_id]
    })

# 5. Build and print the DataFrame cleanly
detailed_metrics_df = pd.DataFrame(rows)

# Quick print confirmation for your train class
train_row = detailed_metrics_df[detailed_metrics_df['Class Name'] == 'train']
if not train_row.empty:
    p_val = train_row['Precision (P)'].values[0]
    r_val = train_row['Recall (R)'].values[0]
    print("=== Target Class Focus ===")
    print(f"Train Class Precision: {p_val:.3f}")
    print(f"Train Class Recall:    {r_val:.3f}\n")

print("=== All Evaluated Classes (Sorted by Lowest Recall First) ===")
display(detailed_metrics_df.sort_values(by='Recall (R)', ascending=True))

=== Target Class Focus ===
Train Class Precision: 0.991
Train Class Recall:    0.885

=== All Evaluated Classes (Sorted by Lowest Recall First) ===


,Class ID,Class Name,Precision (P),Recall (R),mAP@50-95
5,6,road_vehicle,0.686044,0.500000,0.297750
4,5,person,0.906361,0.665001,0.524749
2,3,signal_pole,0.845892,0.812500,0.527550
3,4,signal,0.891498,0.828561,0.631672
1,2,catenary_pole,0.919980,0.867347,0.671406
0,1,train,0.990826,0.885246,0.757423


In [21]:
import os
import pandas as pd
from IPython.display import display

# 1. Find the absolute path to your '3' directory
absolute_path_to_3 = os.path.abspath('3')

# 2. Dynamically get the class names from your loaded model
class_dict = model.names

# 3. Construct a clean combined YAML configuration using your exact folder names
yaml_lines = [
    f"path: {absolute_path_to_3}",
    "train: images/train_split",
    "val:",
    "  - images/train_split",
    "  - images/test_split",
    "names:"
]
for cid, cname in class_dict.items():
    yaml_lines.append(f"  {cid}: {cname}")

combined_yaml_path = '3/combined_data.yaml'
with open(combined_yaml_path, 'w') as f:
    f.write("\n".join(yaml_lines))

print("Successfully generated '3/combined_data.yaml' using train_split and test_split!")
print("Starting evaluation on the COMBINED dataset (Train + Test splits)...")

# 4. Run the validation loop on the combined directories
combined_metrics = model.val(
    data=combined_yaml_path,
    split='val',  # Instructs YOLO to evaluate everything listed under 'val:' above
    conf=0.25,
    iou=0.6,
    plots=True,
    device=0
)

# 5. Extract and align metrics safely to prevent data length mismatch errors
comb_class_names = combined_metrics.names
comb_evaluated_classes = combined_metrics.box.ap_class_index
comb_precisions = combined_metrics.box.p
comb_recalls = combined_metrics.box.r
comb_maps_50_95 = combined_metrics.box.maps

comb_rows = []
for i, class_id in enumerate(comb_evaluated_classes):
    comb_rows.append({
        'Class ID': class_id,
        'Class Name': comb_class_names.get(class_id, f"Class_{class_id}"),
        'Combined Precision (P)': comb_precisions[i],
        'Combined Recall (R)': comb_recalls[i],
        'Combined mAP@50-95': comb_maps_50_95[class_id]
    })

combined_metrics_df = pd.DataFrame(comb_rows)

# Highlight targeted 'train' performance metrics
train_row = combined_metrics_df[combined_metrics_df['Class Name'] == 'train']
if not train_row.empty:
    print("\n=== Combined Train Class Performance ===")
    print(f"Precision: {train_row['Combined Precision (P)'].values[0]:.3f}")
    print(f"Recall:    {train_row['Combined Recall (R)'].values[0]:.3f}")

print("\n=== All Classes Combined Metrics (Sorted by Lowest Recall First) ===")
display(combined_metrics_df.sort_values(by='Combined Recall (R)', ascending=True))

Successfully generated '3/combined_data.yaml' using train_split and test_split!
Starting evaluation on the COMBINED dataset (Train + Test splits)...
Ultralytics 8.4.92 🚀 Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10823MiB)
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 7.0±1.1 MB/s, size: 2083.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /home/jovyan/3/labels/test_split... 498 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 6.2it/s 1:20<0.2sss
val: New cache created: /home/jovyan/3/labels/test_split.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.7it/s 6.8s0.1s
                   all        500       3801      0.828      0.784      0.771       0.56
                 train        480        547      0.977      0.952      0.955      0.849
        

,Class ID,Class Name,Combined Precision (P),Combined Recall (R),Combined mAP@50-95
4,5,person,0.821671,0.653730,0.421478
5,6,road_vehicle,0.783984,0.741176,0.480393
6,7,bicycle,0.679454,0.750000,0.447900
2,3,signal_pole,0.881921,0.770833,0.521932
3,4,signal,0.804014,0.771493,0.563841
1,2,catenary_pole,0.844623,0.845996,0.634262
0,1,train,0.977461,0.952468,0.848909
